# Train Multiple Versions on Kaggle & Track with W&B

**Task**: Fine-tune DistilBERT on AG News (4-class text classification)  
**Platform**: Kaggle (GPU T4 x2)  
**Tracking**: Weights & Biases (W&B)

**Prerequisites**:
- Enable GPU: Settings → Accelerator → GPU T4 x2
- Add Kaggle Secrets: `WANDB_API_KEY`, `HF_TOKEN`, `GITHUB_TOKEN`
- Dataset on HF Hub: [`YuvarajK-g25ait2054/ag-news-cleaned`](https://huggingface.co/datasets/YuvarajK-g25ait2054/ag-news-cleaned)

## Step 0: Install Dependencies

In [ ]:
!pip install -q transformers datasets wandb scikit-learn accelerate

## Step 1: Load Secrets in Kaggle

API keys are stored securely using Kaggle Secrets — never hardcoded.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
import wandb
from huggingface_hub import login

# Load secrets securely
secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('wb_key')
HF_TOKEN = secrets.get_secret('hf_key')
login(token=HF_TOKEN)
wandb.login()

# GitHub token for pushing changes
github_token = secrets.get_secret('GITHUB_TOKEN')

print("Successfully authenticated with W&B, Hugging Face, and GitHub!")

## Step 2: Load Prepared Dataset & Model

In [ ]:
import json
import re
import string
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# load ag_news directly from huggingface - no upload needed
print('Loading ag_news from HuggingFace...')
raw_dataset = load_dataset('ag_news')

# simple text cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

# apply cleaning
dataset = raw_dataset.map(lambda x: {'text': clean_text(x['text']), 'label': x['label']})
print(dataset)

# label mapping
id2label = {
    "0": "World",
    "1": "Sports",
    "2": "Business",
    "3": "Sci/Tech"
}
label2id = {v: int(k) for k, v in id2label.items()}
num_labels = len(id2label)

print(f"\nLabels: {id2label}")
print(f"Number of classes: {num_labels}")

In [ ]:
# Preview the data
print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples:  {len(dataset['test'])}")
dataset["train"].to_pandas().head()

In [ ]:
# Assign train/test splits
train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(f"Train dataset: {train_dataset}")
print(f"Test dataset:  {test_dataset}")

In [ ]:
# Load tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Tokenization complete!")
print(f"Sample features: {train_dataset[0].keys()}")

## Step 3: Define Metrics

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

---
## Step 4: Training Version 1

**Hyperparameters (v1)**:
- Learning rate: `2e-5`
- Batch size: `16`
- Epochs: `3`
- Weight decay: `0.01`

In [ ]:
from transformers import TrainingArguments, Trainer

# Initialise W&B run for Version 1
wandb.init(
    project="mlops-assignment3",
    name="run-v1",
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "max_length": 128,
        "version": "v1",
        "platform": "Kaggle",
    }
)

print("W&B run initialised: run-v1")

In [ ]:
# Load fresh model for v1
model_v1 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Training arguments for v1
training_args_v1 = TrainingArguments(
    output_dir="./results-v1",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name="run-v1",
    logging_steps=100,
)

# Create Trainer
trainer_v1 = Trainer(
    model=model_v1,
    args=training_args_v1,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Train
print("Starting training v1...")
trainer_v1.train()

In [ ]:
# Evaluate v1
results_v1 = trainer_v1.evaluate()
print(f"\n=== Version 1 Results ===")
print(f"  Accuracy: {results_v1['eval_accuracy']:.4f}")
print(f"  F1 Score: {results_v1['eval_f1']:.4f}")
print(f"  Eval Loss: {results_v1['eval_loss']:.4f}")

# Finish W&B run
wandb.finish()
print("\nW&B run-v1 finished.")

---
## Step 5: Training Version 2

**Hyperparameters (v2)** — changed learning rate and batch size:
- Learning rate: `5e-5` (increased from 2e-5)
- Batch size: `32` (increased from 16)
- Epochs: `3`
- Weight decay: `0.01`

In [ ]:
# Initialise W&B run for Version 2
wandb.init(
    project="mlops-assignment3",
    name="run-v2",
    config={
        "model": model_name,
        "epochs": 2, # Reduced for faster training
        "batch_size": 64,
        "learning_rate": 5e-5,
        "weight_decay": 0.01,
        "max_length": 128,
        "version": "v2",
        "platform": "Kaggle",
    }
)

print("W&B run initialised: run-v2")

In [ ]:
# Load fresh model for v2 (start from pretrained, not from v1 checkpoint)
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Training arguments for v2
training_args_v2 = TrainingArguments(
    output_dir="./results-v2",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name="run-v2",
    logging_steps=100,
)

# Create Trainer
trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Train
print("Starting training v2...")
trainer_v2.train()

In [ ]:
# Evaluate v2
results_v2 = trainer_v2.evaluate()
print(f"\n=== Version 2 Results ===")
print(f"  Accuracy: {results_v2['eval_accuracy']:.4f}")
print(f"  F1 Score: {results_v2['eval_f1']:.4f}")
print(f"  Eval Loss: {results_v2['eval_loss']:.4f}")

# Finish W&B run
wandb.finish()
print("\nW&B run-v2 finished.")

---
## Step 6: Compare Results

In [ ]:
# Side-by-side comparison
print("=" * 60)
print("EXPERIMENT COMPARISON")
print("=" * 60)
print(f"{'Metric':<20} {'Version 1':<15} {'Version 2':<15}")
print("-" * 50)
print(f"{'Learning Rate':<20} {'2e-5':<15} {'5e-5':<15}")
print(f"{'Batch Size':<20} {'16':<15} {'32':<15}")
print(f"{'Epochs':<20} {'3':<15} {'3':<15}")
print("-" * 50)
print(f"{'Accuracy':<20} {results_v1['eval_accuracy']:<15.4f} {results_v2['eval_accuracy']:<15.4f}")
print(f"{'F1 (weighted)':<20} {results_v1['eval_f1']:<15.4f} {results_v2['eval_f1']:<15.4f}")
print(f"{'Eval Loss':<20} {results_v1['eval_loss']:<15.4f} {results_v2['eval_loss']:<15.4f}")
print("=" * 60)

# Determine best version
best = "v1" if results_v1['eval_f1'] > results_v2['eval_f1'] else "v2"
print(f"\nBest version: {best}")
print(f"\nCheck your W&B dashboard at: https://wandb.ai/ (project: mlops-assignment3)")

---
## Step 7: Push the Trained Model to Hugging Face Hub

Push the **best** trained model and tokenizer to a public Hugging Face repository so GitHub Actions can load it for inference.

> **Important**: Set the repository visibility to **Public** on Hugging Face before submitting.

In [ ]:
# ---- CHANGE THIS to your Hugging Face username ----
HF_USERNAME = "YuvarajK-g25ait2054"
HF_REPO_NAME = "ag-news-distilbert"
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

# Select the best model based on F1 score
if results_v1["eval_f1"] >= results_v2["eval_f1"]:
    best_trainer = trainer_v1
    best_version = "v1"
else:
    best_trainer = trainer_v2
    best_version = "v2"

print(f"Best version: {best_version}")
print(f"Pushing to: https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# Push model and tokenizer to Hugging Face Hub
best_trainer.model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

print(f"Model and tokenizer pushed to: https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# Log the Hugging Face model URL into W&B run summary
hf_model_url = f"https://huggingface.co/{HF_REPO_ID}"

wandb.init(
    project="mlops-assignment3",
    name=f"push-model-{best_version}",
    config={"best_version": best_version, "hf_repo": HF_REPO_ID}
)
wandb.run.summary["huggingface_model"] = hf_model_url
wandb.run.summary["best_version"] = best_version
wandb.finish()

print(f"Logged HF model URL to W&B summary: {hf_model_url}")

---
## Step 8: Push Notebook & Results to GitHub

Clone the repo, copy the notebook, commit and push to the `ritesh_dev` branch.

In [ ]:
import shutil
import os

# ── GitHub Config ──────────────────────────────────────────────────────────
GITHUB_USERNAME = "riteshmaury-iitj"
REPO_NAME = "group13-assignment-mlops"
BRANCH = "develop"
REPO_PATH = f"/kaggle/working/{REPO_NAME}"

# ── Clone or update repo ──────────────────────────────────────────────────
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
    print("Removed existing repo folder.")

!git clone https://{github_token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git {REPO_PATH}

# Checkout branch (create if it doesn't exist)
!cd {REPO_PATH} && git checkout {BRANCH} 2>/dev/null || git checkout -b {BRANCH}

# ── Configure git ──────────────────────────────────────────────────────────
!git -C {REPO_PATH} config user.email "g25ait2054@iitj.ac.in"
!git -C {REPO_PATH} config user.name "{GITHUB_USERNAME}"

# ── Copy notebook and files to repo ────────────────────────────────────────
import glob

# Copy notebooks
for nb in glob.glob("/kaggle/working/*.ipynb"):
    shutil.copy(nb, REPO_PATH)
    print(f"  Copied: {os.path.basename(nb)}")

# Copy id2label.json if it exists
for f in ["id2label.json"]:
    src = f"/kaggle/working/{f}"
    if os.path.exists(src):
        shutil.copy(src, REPO_PATH)
        print(f"  Copied: {f}")

# ── Commit and push ───────────────────────────────────────────────────────
!cd {REPO_PATH} && git add -A && git status
!cd {REPO_PATH} && git commit -m "Add training notebook with W&B tracking and HF model push"
!cd {REPO_PATH} && git push origin {BRANCH}

print(f"\nPushed to: https://github.com/{GITHUB_USERNAME}/{REPO_NAME}/tree/{BRANCH}")

---
## Notes for Report

1. **Screenshot**: Go to your W&B dashboard → project `mlops-assignment3` → select both runs → take a screenshot showing training loss, validation loss, accuracy, and F1 curves side by side.

2. **Hyperparameter differences**:
   - v1: lr=2e-5, batch_size=16
   - v2: lr=5e-5, batch_size=32

3. **All logged metrics**: training_loss, eval_loss, eval_accuracy, eval_f1 (logged every epoch via `eval_strategy='epoch'`)

4. **Kaggle Secrets used**: `WANDB_API_KEY`, `HF_TOKEN`, `GITHUB_TOKEN` — no tokens hardcoded in the notebook.

5. **Dataset source**: [YuvarajK-g25ait2054/ag-news-cleaned](https://huggingface.co/datasets/YuvarajK-g25ait2054/ag-news-cleaned)